# Aurink 非 LLM 補測:Horizon-base / CodeTed / oliverguhr / ElToluene (2026-11-25)

In [ ]:
# ===== 補測 (修復主測 4 個失敗格) =====
import subprocess, sys, time, torch
subprocess.run([sys.executable,"-m","pip","-q","install","-U","transformers","sentencepiece","huggingface_hub","onnxruntime","fugashi","unidic-lite"], check=False)

FIX_INPUTS = {
 "zh_tw_unpunct": "我們明天早上九點開會記得帶報告還有上個月的數據",
 "zh_tw_filler": "那個我們明天的會議呃改到下午三點然後記得帶那個報告",
 "zh_name_ctl": "我朋友叫做楊知頤他在學校教書",
 "zh_trad_typo": "為了降低少子化，政府可以堆動獎勵生育的政策。",
 "zh_trad_typo2": "我們禮拜三在討論這個問題",
 "zh_simp": "我已经把文件发给你了请注意查收",
 "en_unpunct": "can you send me the report before friday thanks",
 "en_grammar": "their going to review the proposal on Thursday",
 "en_name_ctl": "my colleague Jon will send the invoice for Aurink",
 "ja_unpunct": "明日の会議は十時からです資料を持ってきてください",
 "es_unpunct": "mañana tenemos una reunión a las nueve no olvides traer el informe",
}

def show(name, outs, dt):
    print("="*70); print(f"{name} ({dt:.1f}s)"); print("="*70)
    for k,o in outs:
        print(f"[{k}]\n  IN : {FIX_INPUTS[k]}\n  OUT: {o}")

# --- A. Horizon truecase-base (patch punctuate.py KeyError '0|lower') ---
try:
    from huggingface_hub import hf_hub_download
    import importlib.util
    p=hf_hub_download("Horizon-Labs/multilingual-punctuation-truecase-base","punctuate.py")
    spec=importlib.util.spec_from_file_location("pm2",p); pm2=importlib.util.module_from_spec(spec); spec.loader.exec_module(pm2)
    class _DD(dict):
        def __missing__(self,k): return ""
    pm2.FULLWIDTH=_DD(pm2.FULLWIDTH)
    if hasattr(pm2,"HALFWIDTH"): pm2.HALFWIDTH=_DD(pm2.HALFWIDTH)
    P2=pm2.Punctuator("Horizon-Labs/multilingual-punctuation-truecase-base")
    keys=["zh_tw_unpunct","en_unpunct","en_name_ctl","ja_unpunct","es_unpunct"]
    t0=time.time(); outs=[(k,P2(FIX_INPUTS[k])) for k in keys]; dt=time.time()-t0
    show("Horizon truecase-base (patched)", outs, dt)
except Exception:
    import traceback; traceback.print_exc()

# --- B. CodeTed CSC T5 (direct sentencepiece, 繞過 protobuf 版本問題) ---
try:
    import sentencepiece as spm
    from transformers import T5ForConditionalGeneration
    M="CodeTed/Chinese_Spelling_Correction_T5"
    sp=spm.SentencePieceProcessor(model_file=hf_hub_download(M,"spiece.model"))
    mdl=T5ForConditionalGeneration.from_pretrained(M); mdl.eval()
    def ct5(text):
        ids=sp.encode("糾正句子裡的錯字: "+text)
        e=sp.eos_id()
        if e is not None and e>=0: ids=ids+[e]
        with torch.no_grad(): out=mdl.generate(input_ids=torch.tensor([ids]), max_length=256)
        return sp.decode(out[0].tolist())
    keys=["zh_trad_typo","zh_trad_typo2","zh_simp","zh_name_ctl"]
    t0=time.time(); outs=[(k,ct5(FIX_INPUTS[k])) for k in keys]; dt=time.time()-t0
    show("CodeTed CSC T5 (direct spm)", outs, dt)
except Exception:
    import traceback; traceback.print_exc()

# --- C. oliverguhr spelling (direct load; text2text pipeline 已被 transformers v5 移除) ---
try:
    from transformers import AutoTokenizer, T5ForConditionalGeneration as T5C
    M="oliverguhr/spelling-correction-english-base"
    tok=AutoTokenizer.from_pretrained(M); m3=T5C.from_pretrained(M); m3.eval()
    def sc(text):
        ids=tok(text, return_tensors="pt").input_ids
        with torch.no_grad(): out=m3.generate(ids, max_length=256)
        return tok.decode(out[0], skip_special_tokens=True)
    keys=["en_grammar","en_unpunct","en_name_ctl"]
    t0=time.time(); outs=[(k,sc(FIX_INPUTS[k])) for k in keys]; dt=time.time()-t0
    show("oliverguhr spelling en (direct)", outs, dt)
except Exception:
    import traceback; traceback.print_exc()

# --- D. ElToluene ja punct ---
try:
    from transformers import pipeline
    jp=pipeline("token-classification", model="ElToluene/japanese-punctuation-restoration")
    print("ElToluene raw output [ja_unpunct]:"); print(jp(FIX_INPUTS["ja_unpunct"]))
except Exception:
    import traceback; traceback.print_exc()
